<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 20 — APIs e Deploy com Flask
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — O Oráculo do Titanic Vira uma API
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🌐 Flask + API REST</span>
</div>

## Onde estamos?

| Aula | Tema | Ideia central |
|------|------|---------------|
| 02–03 | EDA e Preparação | Entender e limpar o Titanic |
| 04–09 | Modelos supervisionados | Reg. Linear, Reg. Logística, KNN, SVM, Árvore, Random Forest |
| 10 | K-Means | Encontrar grupos sem rótulos |
| **20** | **API com Flask** | **Tirar o modelo do notebook e colocar no mundo** |

Até hoje, todo modelo que treinamos **nasceu e morreu dentro do notebook**: a
gente treinava, dava um `print()` na acurácia... e acabou. Mas pense no
Random Forest da Aula 09: ele acerta a maioria dos passageiros. Como um
**site**, um **app de celular** ou um **chatbot** poderia usar esse modelo?

> *"Um modelo que só funciona no seu notebook é como um chef que só*
> *cozinha para si mesmo: talentoso, mas ninguém prova a comida."*

Hoje vamos transformar o nosso modelo em uma **API** — um serviço que
qualquer programa pode consultar pela rede, perguntando:
**"essa pessoa sobreviveria ao Titanic?"** 🚢

### A aula tem duas etapas — igual a um projeto de verdade

```
  ☁️ ETAPA 1 — NO COLAB (este notebook)          💻 ETAPA 2 — NO SEU COMPUTADOR (pasta .py)
  ─────────────────────────────────────          ──────────────────────────────────────────
  dados → preparar → treinar → avaliar           app.py         🌐 API Flask ligada
                                 │                                  ▲      │
                                 ▼                                  │      ▼
                     💾 modelo_titanic.pkl  ── download ──►   requisicoes.py / cliente.py
                                                              {"pclass": 3, "sex": "male", ...}
                                                              ◄── {"sobreviveu": 0, "probabilidade": 0.15}
```

Na indústria é assim: o **cientista de dados** treina o modelo em um ambiente
de experimentação (como o Colab) e entrega um **arquivo**; o **sistema** em
produção só carrega esse arquivo e responde aos pedidos.

### Roteiro de hoje

| Parte | Onde | Tema |
|-------|------|------|
| **1** | ☁️ Colab | O que é uma API? — o garçom do restaurante |
| **2** | ☁️ Colab | Congelando o modelo — treinar uma vez, salvar com `pickle` |
| **3** | ☁️ Colab | A receita da API — dos dados crus à previsão (Jack vs Rose) |
| **4** | ☁️ Colab | Baixando o modelo — o `.pkl` sai do Colab |
| **5** | 💻 Computador | A API no ar — Flask, requisições ao vivo e o seu trabalho |

> **Tempo estimado: 45 minutos** · Arquivos da Etapa 2: pasta
> [`Aula_20_API_Flask_Titanic/`](https://github.com/FIAPON/fiap-graduacao-iachatbot/tree/main/Aula_20_API_Flask_Titanic) no GitHub

<div style="background:#d4edda; border-left:5px solid #155724; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#155724;">Por que essa aula importa tanto:</strong>
<span style="color:#155724;"> muitos modelos de Machine Learning nunca saem do notebook de quem os criou. Saber colocar um modelo atrás de uma API é a ponte entre Ciência de Dados e Desenvolvimento de Sistemas — exatamente a sua formação em ADS. É também o que permite que um chatbot use um modelo de IA para responder perguntas — e é o que você vai usar no seu trabalho.</span>
</div>

In [ ]:
import json
import os
import pickle
import time
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split

# ── Carregando e preparando o Titanic (mesma limpeza das aulas anteriores) ────
df = sns.load_dataset("titanic").copy()

df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1
df["sex_enc"]         = (df["sex"] == "female").astype(int)

# As mesmas 5 features usadas desde a Aula 05
FEATURES = ["pclass", "sex_enc", "age", "tamanho_familia", "fare"]

X = df[FEATURES].copy()
y = df["survived"].copy()

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("✅ Dataset pronto!")
print(f"   Treino: {len(X_treino)} passageiros  |  Teste: {len(X_teste)} passageiros")
print(f"   Features usadas: {FEATURES}")

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1 · ☁️ NO COLAB</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">O Que é uma API? — O Garçom do Restaurante</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Você não entra na cozinha do restaurante. Você faz o pedido ao garçom."</p>
    </div>
</div>

**API** (*Application Programming Interface*) é uma forma de um programa
**conversar com outro programa**. A melhor analogia é um restaurante:

| No restaurante 🍽️ | No mundo das APIs 🌐 |
|-------------------|----------------------|
| **Você**, o cliente | O programa que faz o pedido (app, site, **chatbot**) |
| **Cardápio** | A lista de **rotas** disponíveis e o que cada uma espera |
| **Garçom** | A **API** (Flask): recebe o pedido e traz a resposta |
| **Cozinha** | O **modelo de ML**: faz o trabalho pesado, escondido |
| **Pedido anotado** | A **requisição** (*request*), com os dados em JSON |
| **Prato servido** | A **resposta** (*response*), também em JSON |

O cliente **não precisa saber** como a cozinha funciona (se é Random Forest,
KNN ou uma rede neural). Ele só precisa saber **o que pedir** e **o que vai receber**.

### As 3 coisas que você precisa saber sobre HTTP

**1. URL = endereço + rota** — `http://127.0.0.1:5000/prever` → o restaurante
fica em `127.0.0.1:5000` (o seu próprio computador, porta 5000) e o "prato"
pedido é `/prever`.

**2. Métodos** — o *tipo* de pedido:

| Método | Quando usar | No restaurante |
|--------|-------------|----------------|
| `GET` | **Consultar** algo, sem enviar dados | "O que tem no cardápio hoje?" |
| `POST` | **Enviar dados** para serem processados | "Quero este prato, sem cebola, ponto médio" |

**3. Status code** — um número que diz como foi o pedido:

| Código | Significado | No restaurante |
|--------|-------------|----------------|
| `200` | OK, deu tudo certo | Prato servido ✅ |
| `400` | Pedido mal feito (culpa do **cliente**) | "Quero um ... hmm ... coisa" 🤷 |
| `404` | Rota não existe | "Esse prato não está no cardápio" 📋 |
| `500` | Erro interno (culpa do **servidor**) | A cozinha pegou fogo 🔥 |

### JSON: a língua comum

Um programa em Python, outro em JavaScript, outro em Java... como eles se
entendem? Todos falam **JSON** — um formato de **texto** que, para nós,
parece muito com um dicionário Python.

In [ ]:
# JSON é a "língua franca" das APIs: texto puro, que qualquer linguagem entende
passageiro = {"pclass": 3, "sex": "male", "age": 20, "sibsp": 0, "parch": 0, "fare": 7.25}

texto_json = json.dumps(passageiro)          # dicionário Python → texto JSON (para ENVIAR)
print("Tipo antes :", type(passageiro).__name__)
print("Tipo depois:", type(texto_json).__name__)
print("Conteúdo   :", texto_json)

de_volta = json.loads(texto_json)            # texto JSON → dicionário Python (ao RECEBER)
print()
print("De volta a dicionário →  sexo:", de_volta["sex"], "| idade:", de_volta["age"])

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — (a) Para cada situação, diga o método HTTP (GET ou POST) e o status code esperado: <b>(1)</b> o chatbot só quer saber se a API está no ar; <b>(2)</b> o chatbot envia os dados de um passageiro para receber a previsão; <b>(3)</b> o chatbot envia a idade como "vinte" em vez de 20. (b) Por que a comunicação usa JSON (texto) em vez de enviar o objeto Python diretamente?</span></div>

*✏️ (a) (1) `???` / `???`  —  (2) `???` / `???`  —  (3) `???` / `???`*

*✏️ (b) Usamos JSON porque: `???`*

In [ ]:
# ── GABARITO DA MISSÃO 1 (descomente para ver) ───────────────────────────────
# print("(a) (1) GET  / 200 — só consulta, não envia dados")
# print("    (2) POST / 200 — envia os dados do passageiro para processar")
# print("    (3) POST / 400 — o pedido foi mal feito: a culpa é do cliente,")
# print("        então a API deve recusar com uma mensagem explicando o erro")
# print()
# print("(b) Um objeto Python só existe na memória do programa Python. Quem chama")
# print("    a API pode ser um app em Kotlin, um site em JavaScript ou um chatbot")
# print("    em qualquer linguagem. JSON é texto simples e padronizado: todas as")
# print("    linguagens sabem ler e escrever — é a 'língua comum' entre elas.")

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2 · ☁️ NO COLAB</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Congelando o Modelo — Treinar Uma Vez, Servir Mil Vezes</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Treinar é caro; prever é barato. A cozinha prepara a massa de manhã, não a cada pedido."</p>
    </div>
</div>

Vamos colocar em produção o **Random Forest**, campeão do placar da Aula 09.
Ele tem uma vantagem extra para hoje: **não precisa de normalização**, então
é uma peça a menos para levar para a API.

<div style="background:#fff3cd; border-left:5px solid #856404; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#856404;">⚠️ Atenção — o bug mais comum em produção:</strong>
<span style="color:#856404;"> se fôssemos usar KNN, SVM ou Regressão Logística, teríamos que salvar também o <code>StandardScaler</code> <b>ajustado no treino</b> e aplicá-lo na API. Criar um scaler novo na API (ou esquecer dele) faz o modelo "funcionar" sem erro nenhum... mas dar previsões sem sentido. Lembre disso no seu trabalho!</span>
</div>

### O que é `pickle`?

O `pickle` **congela** um objeto Python (com tudo que ele aprendeu) dentro
de um arquivo. Depois, qualquer outro programa pode **descongelar** esse
arquivo e ter o modelo pronto, sem treinar de novo. É como o **save do
videogame** 🎮: você não precisa recomeçar a fase do zero.

```
  COLAB (uma vez)                              API NO COMPUTADOR (sempre que ligar)
  treinar → pickle.dump() ──► 💾 modelo_titanic.pkl ──► pickle.load() → prever, prever, prever...
```

Vamos salvar um **pacote** (dicionário) com o modelo **e** as informações
de que a API vai precisar — inclusive a **ordem** das features e a
**versão do scikit-learn** usada no treino.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ── Treinando o campeão da Aula 09 (mesmos hiperparâmetros) ──────────────────
floresta = RandomForestClassifier(n_estimators=200, max_depth=5,
                                  min_samples_leaf=5, random_state=42)
floresta.fit(X_treino, y_treino)
acuracia = accuracy_score(y_teste, floresta.predict(X_teste))
print(f"🌲 Random Forest treinado — acurácia no teste: {acuracia:.1%}")

# ── Empacotando o modelo junto com o que a API vai precisar ──────────────────
pacote = {
    "modelo": floresta,
    "features": FEATURES,                    # a ORDEM das colunas importa!
    "acuracia_teste": round(acuracia, 4),
    "treinado_em": time.strftime("%Y-%m-%d %H:%M"),
    "versao_sklearn": sklearn.__version__,   # a API vai conferir essa versão
}

with open("modelo_titanic.pkl", "wb") as arquivo:   # "wb" = write binary (escrever)
    pickle.dump(pacote, arquivo)

print(f"💾 Modelo salvo em 'modelo_titanic.pkl' ({os.path.getsize('modelo_titanic.pkl') / 1024:.0f} KB)")

In [ ]:
# "Descongelando" — é exatamente isso que o app.py vai fazer quando ligar
with open("modelo_titanic.pkl", "rb") as arquivo:   # "rb" = read binary (ler)
    pacote_carregado = pickle.load(arquivo)

modelo_carregado = pacote_carregado["modelo"]
iguais = (modelo_carregado.predict(X_teste) == floresta.predict(X_teste)).all()

print("Conteúdo do pacote:", list(pacote_carregado.keys()))
print("O modelo carregado prevê exatamente igual ao original?", "✅ Sim" if iguais else "❌ Não")

### Por que guardar a ordem das features?

O modelo não sabe o **nome** das colunas quando recebe só números — ele sabe
apenas que "a 1ª coluna é X, a 2ª é Y...". Veja o que acontece com a
**Rose** (do filme) quando trocamos a ordem das colunas:

In [ ]:
# Rose: 1ª classe, mulher, 17 anos, viajava com a mãe (família = 2), passagem cara
rose_preparada = pd.DataFrame([{"pclass": 1, "sex_enc": 1, "age": 17,
                                "tamanho_familia": 2, "fare": 151.55}])

ordem_errada = ["age", "fare", "pclass", "sex_enc", "tamanho_familia"]

p_certa  = floresta.predict_proba(rose_preparada[FEATURES].values)[0, 1]
p_errada = floresta.predict_proba(rose_preparada[ordem_errada].values)[0, 1]

print(f"Ordem certa  {FEATURES}")
print(f"   → chance de sobreviver: {p_certa:.0%}")
print(f"Ordem errada {ordem_errada}")
print(f"   → chance de sobreviver: {p_errada:.0%}   😱 nenhum erro apareceu!")

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — (a) Por que a API vai carregar o modelo do arquivo <code>.pkl</code> em vez de treinar o Random Forest de novo a cada pedido? (b) No experimento acima, o Python mostrou algum erro quando a ordem das colunas estava errada? Por que isso é perigoso em produção?</span></div>

*✏️ (a) Não treinamos a cada pedido porque: `???`*

*✏️ (b) Apareceu erro? `???` — Isso é perigoso porque: `???`*

In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# print("(a) Treinar é lento e caro (200 árvores!), e cada pedido levaria segundos.")
# print("    Além disso, o resultado mudaria se os dados mudassem, e a API precisaria")
# print("    ter o dataset inteiro disponível. Carregando o .pkl, o modelo já vem")
# print("    pronto: a previsão leva milissegundos e é sempre a mesma versão do modelo.")
# print()
# print("(b) Nenhum erro! O modelo recebeu 5 números e fez a conta — só que tratou")
# print("    a idade (17) como se fosse a classe, a tarifa como idade, etc. É um")
# print("    'bug silencioso': a API responde normalmente, mas a previsão é lixo.")
# print("    Por isso salvamos a lista FEATURES junto do modelo e montamos as")
# print("    colunas na API sempre nessa ordem.")

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3 · ☁️ NO COLAB</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">A Receita da API — Dos Dados Crus à Previsão</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Antes de abrir o restaurante, o chef testa a receita na própria cozinha."</p>
    </div>
</div>

Quem usa a API (um app, um chatbot) envia os dados **"crus"**, do jeito que
um usuário conhece: classe, sexo, idade, irmãos, pais, tarifa. Mas o modelo
foi treinado com `sex_enc` e `tamanho_familia`!

<div style="background:#d1ecf1; border-left:5px solid #0c5460; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#0c5460;">A ideia mais importante da aula:</strong>
<span style="color:#0c5460;"> o usuário não sabe o que é <code>sex_enc</code> ou <code>tamanho_familia</code> — e nem deveria precisar saber! A API é que precisa aplicar <b>exatamente a mesma preparação</b> que fizemos no treino (Aula 03). Quem faz isso é a função <code>preparar_passageiro()</code>.</span>
</div>

```
 dados crus (o que o cliente envia)          preparar_passageiro()              o modelo recebe
 ──────────────────────────────────          ─────────────────────              ───────────────
 {"pclass": 1, "sex": "female",        ───►  sex → sex_enc                ───►  pclass  sex_enc  age  tamanho_familia  fare
  "age": 17, "sibsp": 0, "parch": 1,         sibsp + parch + 1                    1       1      17         2         151.55
  "fare": 151.55}                            colunas na ordem FEATURES
```

As duas funções abaixo são **as mesmas** que estão no `app.py` da Parte 5.
Testando aqui no Colab, temos certeza de que a "receita" funciona antes de
abrir o restaurante.

In [ ]:
# ── O mesmo que o app.py faz ao ligar: carrega o modelo e as features do pacote ──
modelo = pacote_carregado["modelo"]
FEATURES = pacote_carregado["features"]

# Campos que o cliente precisa enviar (dados "crus", como o usuário conhece)
CAMPOS_OBRIGATORIOS = ["pclass", "sex", "age", "sibsp", "parch", "fare"]


def preparar_passageiro(dados):
    """Repete exatamente a mesma preparação feita no treino (Aula 03)."""
    faltando = [campo for campo in CAMPOS_OBRIGATORIOS if campo not in dados]
    if faltando:
        raise ValueError(f"Campos obrigatórios faltando: {faltando}")

    try:
        passageiro = {
            "pclass": int(dados["pclass"]),
            "sex_enc": 1 if dados["sex"] == "female" else 0,
            "age": float(dados["age"]),
            "tamanho_familia": int(dados["sibsp"]) + int(dados["parch"]) + 1,
            "fare": float(dados["fare"]),
        }
    except (ValueError, TypeError):
        raise ValueError("pclass, age, sibsp, parch e fare precisam ser números.") from None

    return pd.DataFrame([passageiro])[FEATURES]      # mesma ordem de colunas do treino


def prever_passageiro(dados):
    """Dados crus → previsão, já no formato da resposta da API."""
    X = preparar_passageiro(dados)
    sobreviveu = int(modelo.predict(X)[0])
    probabilidade = float(modelo.predict_proba(X)[0, 1])
    return {
        "sobreviveu": sobreviveu,
        "probabilidade_sobrevivencia": round(probabilidade, 4),
        "mensagem": "🛟 Conseguiu um lugar no bote!" if sobreviveu else "🧊 Não resistiu ao naufrágio...",
    }

### 🎬 Jack vs Rose

Vamos consultar o Oráculo sobre os dois protagonistas do filme *Titanic* (1997):

| | Jack Dawson 🃏 | Rose DeWitt Bukater 💎 |
|---|---|---|
| Classe | 3ª (ganhou a passagem no pôquer) | 1ª |
| Sexo | masculino | feminino |
| Idade | 20 | 17 |
| A bordo com ele(a) | ninguém | a mãe (`parch = 1`) |
| Tarifa | barata | cara |

In [ ]:
jack = {"pclass": 3, "sex": "male",   "age": 20, "sibsp": 0, "parch": 0, "fare": 7.75}
rose = {"pclass": 1, "sex": "female", "age": 17, "sibsp": 0, "parch": 1, "fare": 151.55}

for nome, passageiro in [("Jack", jack), ("Rose", rose)]:
    r = prever_passageiro(passageiro)
    print(f"{nome:<5} → {r['mensagem']}  (probabilidade: {r['probabilidade_sobrevivencia']:.0%})")

print()
print("Resposta completa da Rose — é exatamente esse JSON que a API vai devolver:")
print(json.dumps(r, indent=2, ensure_ascii=False))

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — <b>E você, sobreviveria?</b> (a) Preencha <code>meu_passageiro</code> com os seus dados (ou de um personagem inventado). Qual foi a sua probabilidade? (b) Rode o experimento seguinte, que muda <b>só a classe</b> (e a tarifa típica dela). Quanto a probabilidade muda da 1ª para a 3ª classe? O que isso diz sobre o Titanic?</span></div>

In [ ]:
# ✏️ MISSÃO 3 (a) — troque pelos SEUS dados
meu_passageiro = {
    "pclass": 2,        # 1 = primeira, 2 = segunda, 3 = terceira classe
    "sex": "female",    # "male" ou "female"
    "age": 21,
    "sibsp": 0,         # nº de irmãos/cônjuge a bordo
    "parch": 0,         # nº de pais/filhos a bordo
    "fare": 19.0,       # tarifa típica: 1ª ~ 62 | 2ª ~ 19 | 3ª ~ 8
}

r = prever_passageiro(meu_passageiro)
print(f"🔮 O Oráculo diz: {r['mensagem']}  (probabilidade: {r['probabilidade_sobrevivencia']:.0%})")

In [ ]:
# MISSÃO 3 (b) — mesmo passageiro, só mudando a classe (com a tarifa mediana de cada uma)
tarifa_tipica = df.groupby("pclass")["fare"].median()

for classe in [1, 2, 3]:
    teste = {**meu_passageiro, "pclass": classe, "fare": float(tarifa_tipica[classe])}
    r = prever_passageiro(teste)
    barra = "█" * round(r["probabilidade_sobrevivencia"] * 30)
    print(f"  {classe}ª classe (tarifa {tarifa_tipica[classe]:>5.1f}) → "
          f"{r['probabilidade_sobrevivencia']:>4.0%} {barra}")

*✏️ (a) Minha probabilidade de sobreviver foi: `???`*

*✏️ (b) Da 1ª para a 3ª classe a probabilidade mudou de `???` para `???`. Isso mostra que: `???`*

In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# print("(a) Resposta pessoal — cada um terá um resultado diferente!")
# print()
# print("(b) A probabilidade cai bastante da 1ª para a 3ª classe, mesmo mantendo")
# print("    sexo, idade e família iguais. O modelo aprendeu com os dados reais")
# print("    que a classe social pesou na sobrevivência: os passageiros da 1ª")
# print("    classe estavam mais perto dos botes e tiveram prioridade no embarque.")
# print("    O que vimos na EDA da Aula 02 agora está 'embutido' na API.")

### Quando o cliente erra

Uma API pública recebe pedidos de qualquer lugar — e o usuário **sempre** vai
mandar algo que você não esperava. A nossa função se defende levantando um
`ValueError` com uma mensagem clara. Na API, esse erro vai virar uma resposta
com **status `400`** (pedido mal feito).

In [ ]:
pedidos_ruins = {
    "Esqueceu a idade":  {k: v for k, v in jack.items() if k != "age"},
    "Idade por extenso": {**jack, "age": "vinte"},
}

for descricao, passageiro in pedidos_ruins.items():
    try:
        prever_passageiro(passageiro)
    except ValueError as erro:
        print(f"{descricao:<18} → ❌ recusado (na API: status 400) — {erro}")

Ótimo, a função recusou os dois. 👏 Mas existe um erro mais traiçoeiro...
Veja o que acontece se o app de alguém mandar a Rose com `"Female"`
(com F maiúsculo):

In [ ]:
rose_maiuscula = {**rose, "sex": "Female"}   # mesma Rose, só com F maiúsculo

for descricao, passageiro in [('"female"', rose), ('"Female"', rose_maiuscula)]:
    r = prever_passageiro(passageiro)
    print(f"Rose com sex={descricao:<9} → {r['mensagem']}  "
          f"(probabilidade: {r['probabilidade_sobrevivencia']:.0%})")

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 — (a) Nenhum erro apareceu para a Rose com <code>"Female"</code>, mas a previsão mudou. Olhando a função <code>preparar_passageiro()</code>, explique por quê. (b) Corrija a função na célula da Parte 3: converta o sexo para minúsculas e levante um <code>ValueError</code> se ele não for <code>"male"</code> nem <code>"female"</code>. Rode a célula de novo e teste com a célula abaixo.</span></div>

In [ ]:
# ✏️ Teste da Missão 4 — depois da correção, "Female" deve funcionar e "mulher" deve ser recusado
for valor in ["female", "Female", "FEMALE", "mulher"]:
    try:
        r = prever_passageiro({**rose, "sex": valor})
        print(f"sex={valor!r:<9} → {r['mensagem']}  (probabilidade: {r['probabilidade_sobrevivencia']:.0%})")
    except ValueError as erro:
        print(f"sex={valor!r:<9} → ❌ recusado — {erro}")

*✏️ (a) A previsão mudou porque: `???`*

*✏️ (b) Código que adicionei na função: `???`*

In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# print("(a) Em preparar_passageiro() temos: 1 if dados['sex'] == 'female' else 0")
# print("    Como 'Female' != 'female', a comparação dá False e a Rose vira")
# print("    sex_enc = 0 (homem)! Nenhum erro aparece: é um bug silencioso,")
# print("    o tipo mais perigoso — o cliente confia numa resposta errada.")
# print()
# print("(b) Logo depois da checagem de campos faltando, acrescente:")
# print("""
#     sexo = str(dados["sex"]).strip().lower()         # aceita "Female", "MALE"...
#     if sexo not in ("male", "female"):               # ...e recusa "mulher"
#         raise ValueError("O campo 'sex' precisa ser 'male' ou 'female'.")
# """)
# print("    e troque dados['sex'] por sexo na linha do sex_enc.")
# print("    Essa é exatamente a versão que está no app.py da Parte 5.")

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4 · ☁️ NO COLAB</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Baixando o Modelo — O .pkl Sai do Colab</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"A receita está testada. Agora ela vai para o restaurante."</p>
    </div>
</div>

O arquivo `modelo_titanic.pkl` foi salvo na máquina do Google, não no seu
computador. A célula abaixo faz o **download** dele (no Colab, também dá para
achá-lo no ícone de 📁 pasta, na barra lateral esquerda).

<div style="background:#fff3cd; border-left:5px solid #856404; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#856404;">⚠️ Anote a versão do scikit-learn!</strong>
<span style="color:#856404;"> um <code>.pkl</code> deve ser aberto com a <b>mesma versão</b> do scikit-learn usada no treino. Com versões diferentes, ele pode dar aviso, erro ou — pior — previsões diferentes. A célula abaixo mostra a versão do Colab; no seu computador, instale a mesma com <code>pip install scikit-learn==X.Y.Z</code>.</span>
</div>

In [ ]:
print("📦 Versões usadas no treino (instale as mesmas no seu computador):")
print(f"   scikit-learn == {sklearn.__version__}")
print(f"   pandas       == {pd.__version__}")
print()

try:
    from google.colab import files               # só existe dentro do Colab
    files.download("modelo_titanic.pkl")
    print("⬇️  Download iniciado! Procure o arquivo na pasta de Downloads.")
except ImportError:
    print(f"💻 Fora do Colab: o arquivo já está em {os.path.abspath('modelo_titanic.pkl')}")

print()
print("Próximo passo: coloque o modelo_titanic.pkl na pasta Aula_20_API_Flask_Titanic/,")
print("ao lado do app.py — e siga para a Parte 5.")

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5 · 💻 NO SEU COMPUTADOR</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">A API no Ar — Flask e Requisições ao Vivo</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Com poucas linhas de código, seu computador vira um servidor."</p>
    </div>
</div>

A partir daqui, saímos do Colab e vamos para o **seu computador**. Os
arquivos estão na pasta
[`Aula_20_API_Flask_Titanic/`](https://github.com/FIAPON/fiap-graduacao-iachatbot/tree/main/Aula_20_API_Flask_Titanic) do repositório:

| Arquivo | Papel no restaurante 🍽️ |
|---------|--------------------------|
| `modelo_titanic.pkl` | A receita pronta — você acabou de baixar na Parte 4 |
| `app.py` | O **garçom**: a API Flask que recebe pedidos e devolve previsões |
| `requisicoes.py` | Uma demonstração que dispara os pedidos **um a um** |
| `cliente.py` | Um **mini-chatbot** de terminal que pergunta seus dados e consulta a API |

### Anatomia de uma API Flask

**Flask** é um *microframework* web em Python: leve, simples e muito usado
para colocar modelos de ML no ar. O `app.py` segue esta estrutura:

```python
app = Flask(__name__)                        # 1. cria a aplicação (abre o restaurante)

@app.route("/")                              # 2. cria uma ROTA (um item do cardápio)
def home():                                  # 3. função que roda quando alguém acessa a rota
    return jsonify({"mensagem": "Olá!"})     # 4. a resposta, em JSON

@app.route("/prever", methods=["POST"])      # 5. rota que RECEBE dados
def prever():
    dados = request.get_json()               #    lê o JSON enviado pelo cliente
    try:
        resultado = prever_passageiro(dados) #    a MESMA função da Parte 3!
    except ValueError as erro:
        return jsonify({"erro": str(erro)}), 400   # pedido mal feito → status 400
    return jsonify(resultado)                #    tudo certo → status 200

app.run(port=5000)                           # 6. liga o servidor e fica esperando pedidos
```

O `@app.route(...)` é um **decorador**: uma "etiqueta" que liga um endereço
(URL) a uma função Python. Repare que o **modelo é carregado uma única vez**,
quando o servidor liga — não a cada pedido.

### Passo a passo no seu computador

```bash
# 0. na pasta Aula_20_API_Flask_Titanic/, com o modelo_titanic.pkl dentro dela
pip install -r requirements.txt
pip install scikit-learn==X.Y.Z      # a versão que a Parte 4 mostrou

# 1. TERMINAL 1 — liga a API (e deixa aberto!)
python app.py

# 2. TERMINAL 2 — faz as requisições
python requisicoes.py                # demonstração: um pedido a cada Enter
python cliente.py                    # mini-chatbot: e você, sobreviveria?
```

Deixe os **dois terminais lado a lado**: cada pedido que sai do
`requisicoes.py` aparece na hora no terminal do `app.py`, com o status code:

```
  TERMINAL 2 — requisicoes.py                       TERMINAL 1 — app.py
  ───────────────────────────                       ───────────────────
  ➡️  POST http://127.0.0.1:5000/prever
      corpo: {"pclass": 3, "sex": "male", ...}
  ⬅️  ✅ [200] {"mensagem": "🧊 Não resistiu..."}    "POST /prever HTTP/1.1" 200
  ...
  ➡️  POST http://127.0.0.1:5000/prever
      corpo: {..., "sex": "mulher"}
  ⬅️  ❌ [400] {"erro": "O campo 'sex' precisa..."}  "POST /prever HTTP/1.1" 400
  ...
  ➡️  GET http://127.0.0.1:5000/cozinha
  ⬅️  ❌ [404] (página de erro do Flask)             "GET /cozinha HTTP/1.1" 404
```

Com o servidor ligado, as rotas GET também abrem direto no **navegador**:
http://127.0.0.1:5000/ e http://127.0.0.1:5000/info

<div style="background:#e3f2fd; border-left:5px solid #1565c0; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#1565c0;">Por que não rodar a API no Colab?</strong>
<span style="color:#1565c0;"> o Colab roda numa máquina do Google: um servidor ligado lá não é acessível pelo navegador do seu computador nem pelos apps dos seus colegas. Rodando localmente, você vê a API funcionando de verdade — o mesmo jeito que ela rodaria em um servidor de produção.</span>
</div>

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 5 (final, no seu computador) — (a) Com a API ligada, abra http://127.0.0.1:5000/tripulacao no navegador. Qual status code aparece no terminal do <code>app.py</code>? (b) Adicione ao <code>app.py</code> uma rota GET <code>/tripulacao</code> que devolva <code>{"capitao": "Edward Smith", "navio": "RMS Titanic"}</code>. Desligue o servidor (<code>Ctrl+C</code>), ligue de novo e recarregue a página. O que mudou?</span></div>

*✏️ (a) Antes de criar a rota, o status foi: `???` — porque: `???`*

*✏️ (b) Depois de criar a rota e religar o servidor: `???`*

In [ ]:
# ── GABARITO DA MISSÃO 5 (descomente para ver) ───────────────────────────────
# print("(a) 404 (Not Found) — a rota não estava no 'cardápio' da API.")
# print()
# print("(b) Basta acrescentar no app.py, junto das outras rotas:")
# print("""
# @app.route("/tripulacao")
# def tripulacao():
#     return jsonify({"capitao": "Edward Smith", "navio": "RMS Titanic"})
# """)
# print("    Depois de religar, o navegador mostra o JSON e o terminal do app.py")
# print("    registra status 200. Lembre: o servidor precisa ser RELIGADO para")
# print("    ler o arquivo novo!")

---

## 🛠️ Levando para o seu trabalho

O `app.py` foi escrito para ser reaproveitado. Para colocar o **seu** modelo
atrás de uma API, siga o mesmo caminho desta aula:

| Onde | O que fazer |
|------|-------------|
| ☁️ Seu notebook | Treine e salve um **pacote** com o modelo, a lista de features e a versão do scikit-learn (Parte 2). Usou normalização? **Salve o scaler junto!** |
| ☁️ Seu notebook | Escreva e teste a função de **preparação** dos dados crus (Parte 3) |
| ☁️ Seu notebook | Baixe o `.pkl` e anote a versão do scikit-learn (Parte 4) |
| 💻 `app.py` | Mexa só no bloco **`✏️ ADAPTE AQUI`**: nome do `.pkl`, campos obrigatórios, `preparar_passageiro()` e `prever_passageiro()` |
| 💻 `requisicoes.py` | Troque os exemplos pelos dados do seu problema e mostre as requisições |

O passo a passo completo está no
[README da pasta](https://github.com/FIAPON/fiap-graduacao-iachatbot/tree/main/Aula_20_API_Flask_Titanic).

### E o deploy "de verdade"?

O `app.run()` liga um **servidor de desenvolvimento** — ótimo para aprender,
mas não para milhares de usuários. Em produção, a mesma API roda com um
servidor robusto (como `gunicorn` ou `waitress`) em um serviço de nuvem
(Render, Railway, AWS, Azure, Google Cloud...), que dá a ela um **endereço
público** na internet, como `https://oraculo-titanic.onrender.com/prever`.
O código Flask continua praticamente o mesmo!

### E o chatbot? 🤖

É aqui que as duas metades da disciplina se encontram. Um chatbot pode
**conversar** com o usuário, coletar as informações e **chamar a nossa API**
por trás dos panos:

```
👤 "Oi! Eu sobreviveria ao Titanic?"
🤖 "Vamos descobrir! Em qual classe você viajaria?"   ← coleta os dados na conversa
👤 "Na terceira, tenho 20 anos, viajaria sozinho"
         │
         └──► POST /prever {"pclass": 3, "sex": "male", "age": 20, ...}
                  ◄── {"sobreviveu": 0, "probabilidade_sobrevivencia": 0.15}
🤖 "Sinto muito... você teria cerca de 15% de chance. Que tal a 1ª classe? 😅"
```

O `cliente.py` da pasta é exatamente uma versão simples disso!

**✏️ Minha reflexão sobre a aula:**

1. API em uma frase simples: *...*

2. A API precisa repetir a preparação dos dados do treino porque: *...*

3. No meu trabalho, a minha API vai receber estes dados e devolver esta resposta: *...*